In [1]:
# Advanced Biomedical Signal Processing Platform
# Complete implementation with deep learning, real-time processing, and clinical analysis

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from scipy.fft import fft, fftfreq
import pywt
from PyEMD import EMD, EEMD
import heartpy as hp
import neurokit2 as nk
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, LSTM, GRU, Dense,
                                   Dropout, BatchNormalization, Attention,
                                   Input, Concatenate, GlobalAveragePooling1D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import streamlit as st
from datetime import datetime, timedelta
import threading
import queue
import warnings
import json
import pickle
from pathlib import Path
warnings.filterwarnings('ignore')

# ======================= ADVANCED SIGNAL PROCESSING CLASS =======================

class AdvancedBioSignalProcessor:
    """Advanced biomedical signal processing with deep learning capabilities"""

    def __init__(self, sampling_rate=500):
        self.fs = sampling_rate
        self.models = {}
        self.scalers = {}
        self.feature_names = []

    def generate_synthetic_ecg(self, duration=60, noise_level=0.05, arrhythmia_type=None):
        """Generate synthetic ECG data with controllable arrhythmias"""
        t = np.linspace(0, duration, int(duration * self.fs))
        ecg = np.zeros_like(t)

        # Base heart rate with variability
        base_hr = 70
        hr_variability = 10 * np.sin(2 * np.pi * 0.1 * t)  # HRV
        instantaneous_hr = base_hr + hr_variability

        # Generate R-R intervals
        rr_intervals = 60 / instantaneous_hr

        # Create ECG morphology
        for i, rr in enumerate(rr_intervals[:-1]):
            peak_time = np.sum(rr_intervals[:i+1])
            if peak_time < duration:
                # P wave
                p_center = peak_time - 0.15
                if p_center > 0:
                    p_wave = 0.1 * np.exp(-((t - p_center) / 0.02)**2)
                    ecg += p_wave

                # QRS complex
                qrs_duration = 0.08
                qrs_kernel = signal.gaussian(int(qrs_duration * self.fs), std=0.01 * self.fs)
                qrs_start = int((peak_time - qrs_duration/2) * self.fs)
                qrs_end = min(qrs_start + len(qrs_kernel), len(ecg))
                if qrs_start >= 0 and qrs_start < len(ecg):
                    ecg[qrs_start:qrs_end] += qrs_kernel[:qrs_end-qrs_start] * 1.2

                # T wave
                t_center = peak_time + 0.3
                if t_center < duration:
                    t_wave = 0.3 * np.exp(-((t - t_center) / 0.1)**2)
                    ecg += t_wave

        # Add arrhythmias
        if arrhythmia_type == 'atrial_fibrillation':
            # Irregular R-R intervals and absent P waves
            af_modulation = 0.3 * np.random.randn(len(t))
            ecg += af_modulation

        elif arrhythmia_type == 'ventricular_tachycardia':
            # Wide QRS complexes at high rate
            vt_frequency = 3.5  # 210 bpm
            vt_component = 0.8 * np.sin(2 * np.pi * vt_frequency * t)
            ecg += vt_component

        elif arrhythmia_type == 'premature_ventricular_contraction':
            # Add PVCs at random intervals
            pvc_times = np.random.choice(t[int(len(t)*0.1):-int(len(t)*0.1)],
                                       size=int(duration/10), replace=False)
            for pvc_time in pvc_times:
                pvc_idx = int(pvc_time * self.fs)
                if pvc_idx < len(ecg) - 100:
                    # Wide, bizarre QRS
                    pvc_morphology = signal.gaussian(200, std=50) * 1.5
                    ecg[pvc_idx:pvc_idx+200] += pvc_morphology

        # Add realistic noise
        baseline_wander = 0.1 * np.sin(2 * np.pi * 0.5 * t)
        muscle_noise = noise_level * np.random.randn(len(t))
        powerline_noise = 0.02 * np.sin(2 * np.pi * 50 * t)

        ecg = ecg + baseline_wander + muscle_noise + powerline_noise

        return ecg, t

    def advanced_preprocessing(self, signal_data, signal_type='ecg'):
        """Advanced preprocessing pipeline with multiple methods"""

        # Step 1: Baseline correction using empirical mode decomposition
        emd = EEMD()
        imfs = emd.eemd(signal_data, max_imf=8)

        # Remove slow varying components (baseline drift)
        reconstructed = np.sum(imfs[:-2], axis=0)

        # Step 2: Adaptive filtering
        if signal_type == 'ecg':
            # ECG-specific filtering
            # Butterworth bandpass filter
            sos = signal.butter(4, [0.5, 40], btype='band', fs=self.fs, output='sos')
            filtered_signal = signal.sosfilt(sos, reconstructed)

            # Notch filter for powerline interference
            b_notch, a_notch = signal.iirnotch(50, 30, fs=self.fs)
            filtered_signal = signal.filtfilt(b_notch, a_notch, filtered_signal)

        elif signal_type == 'eeg':
            # EEG-specific filtering
            sos = signal.butter(4, [0.1, 50], btype='band', fs=self.fs, output='sos')
            filtered_signal = signal.sosfilt(sos, reconstructed)

        # Step 3: Wavelet denoising
        coeffs = pywt.wavedec(filtered_signal, 'db8', level=6)

        # Soft thresholding
        threshold = 0.1 * np.max(np.abs(coeffs[0]))
        coeffs_thresh = [pywt.threshold(c, threshold, mode='soft') for c in coeffs]
        denoised_signal = pywt.waverec(coeffs_thresh, 'db8')

        return denoised_signal

    def extract_comprehensive_features(self, signal_data, signal_type='ecg'):
        """Extract comprehensive features for machine learning"""
        features = {}

        # Time domain features
        features['mean'] = np.mean(signal_data)
        features['std'] = np.std(signal_data)
        features['var'] = np.var(signal_data)
        features['rms'] = np.sqrt(np.mean(signal_data**2))
        features['skewness'] = stats.skew(signal_data)
        features['kurtosis'] = stats.kurtosis(signal_data)
        features['zero_crossings'] = len(np.where(np.diff(np.sign(signal_data)))[0])

        # Frequency domain features
        freqs, psd = signal.welch(signal_data, fs=self.fs, nperseg=1024)
        features['spectral_centroid'] = np.sum(freqs * psd) / np.sum(psd)
        features['spectral_bandwidth'] = np.sqrt(np.sum((freqs - features['spectral_centroid'])**2 * psd) / np.sum(psd))
        features['spectral_rolloff'] = freqs[np.where(np.cumsum(psd) >= 0.85 * np.sum(psd))[0][0]]

        # Power in different frequency bands
        if signal_type == 'ecg':
            bands = {'low': (0.5, 5), 'mid': (5, 15), 'high': (15, 40)}
        else:  # EEG
            bands = {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 13),
                    'beta': (13, 30), 'gamma': (30, 50)}

        for band_name, (low, high) in bands.items():
            band_mask = (freqs >= low) & (freqs <= high)
            features[f'power_{band_name}'] = np.sum(psd[band_mask])

        # Wavelet features
        scales = np.arange(1, 32)
        coefficients, _ = pywt.cwt(signal_data, scales, 'morl')
        features['wavelet_energy'] = np.sum(np.abs(coefficients)**2)
        features['wavelet_entropy'] = -np.sum(np.abs(coefficients)**2 * np.log(np.abs(coefficients)**2 + 1e-12))

        # Non-linear features
        # Approximate entropy
        features['approximate_entropy'] = self.approximate_entropy(signal_data, m=2, r=0.2*np.std(signal_data))

        # Sample entropy
        features['sample_entropy'] = self.sample_entropy(signal_data, m=2, r=0.2*np.std(signal_data))

        # Detrended fluctuation analysis
        features['dfa_alpha'] = self.detrended_fluctuation_analysis(signal_data)

        if signal_type == 'ecg':
            # ECG-specific features
            ecg_features = self.extract_ecg_features(signal_data)
            features.update(ecg_features)

        return features

    def extract_ecg_features(self, ecg_signal):
        """Extract ECG-specific clinical features"""
        features = {}

        # R-peak detection using multiple methods
        try:
            # Method 1: NeuroKit2
            peaks_nk = nk.ecg_peaks(ecg_signal, sampling_rate=self.fs)[1]['ECG_R_Peaks']

            if len(peaks_nk) > 2:
                # Heart rate statistics
                rr_intervals = np.diff(peaks_nk) / self.fs * 1000  # ms
                features['mean_hr'] = 60000 / np.mean(rr_intervals)
                features['std_hr'] = np.std(60000 / rr_intervals)

                # HRV time domain
                features['mean_rr'] = np.mean(rr_intervals)
                features['std_rr'] = np.std(rr_intervals)
                features['rmssd'] = np.sqrt(np.mean(np.diff(rr_intervals)**2))
                features['pnn50'] = np.sum(np.abs(np.diff(rr_intervals)) > 50) / len(rr_intervals) * 100

                # HRV frequency domain
                if len(rr_intervals) > 10:
                    # Interpolate RR series
                    time_rr = np.cumsum(rr_intervals) / 1000
                    time_interp = np.arange(0, time_rr[-1], 1/4)  # 4 Hz
                    rr_interp = np.interp(time_interp, time_rr, rr_intervals)

                    # Power spectral density
                    freqs_hrv, psd_hrv = signal.welch(rr_interp, fs=4, nperseg=min(256, len(rr_interp)))

                    # HRV frequency bands
                    vlf_mask = (freqs_hrv >= 0.0033) & (freqs_hrv < 0.04)
                    lf_mask = (freqs_hrv >= 0.04) & (freqs_hrv < 0.15)
                    hf_mask = (freqs_hrv >= 0.15) & (freqs_hrv < 0.4)

                    features['vlf_power'] = np.sum(psd_hrv[vlf_mask])
                    features['lf_power'] = np.sum(psd_hrv[lf_mask])
                    features['hf_power'] = np.sum(psd_hrv[hf_mask])
                    features['lf_hf_ratio'] = features['lf_power'] / (features['hf_power'] + 1e-6)

                # Morphological features
                for i, peak in enumerate(peaks_nk[1:-1], 1):  # Skip first and last
                    start_idx = max(0, peak - int(0.2 * self.fs))
                    end_idx = min(len(ecg_signal), peak + int(0.4 * self.fs))
                    beat = ecg_signal[start_idx:end_idx]

                    if len(beat) > 0:
                        if i == 1:  # Initialize with first beat
                            features['qrs_width'] = len(beat)
                            features['r_amplitude'] = np.max(beat)
                            features['q_amplitude'] = np.min(beat[:len(beat)//3])
                            features['s_amplitude'] = np.min(beat[len(beat)//3:])

        except Exception as e:
            print(f"ECG feature extraction error: {e}")
            # Fill with default values
            for key in ['mean_hr', 'std_hr', 'mean_rr', 'std_rr', 'rmssd', 'pnn50',
                       'vlf_power', 'lf_power', 'hf_power', 'lf_hf_ratio']:
                features[key] = 0.0

        return features

    def approximate_entropy(self, data, m, r):
        """Calculate approximate entropy"""
        def _maxdist(xi, xj):
            return max([abs(ua - va) for ua, va in zip(xi, xj)])

        def _phi(m):
            patterns = np.array([data[i:i + m] for i in range(len(data) - m + 1)])
            C = np.zeros(len(patterns))
            for i in range(len(patterns)):
                template_i = patterns[i]
                for j in range(len(patterns)):
                    if _maxdist(template_i, patterns[j]) <= r:
                        C[i] += 1.0
            phi = (1.0 / len(patterns)) * sum(np.log(C / len(patterns)))
            return phi

        return _phi(m) - _phi(m + 1)

    def sample_entropy(self, data, m, r):
        """Calculate sample entropy"""
        def _maxdist(xi, xj):
            return max([abs(ua - va) for ua, va in zip(xi, xj)])

        N = len(data)
        B = 0.0
        A = 0.0

        # Template of length m
        for i in range(N - m):
            template = data[i:i + m]
            for j in range(i + 1, N - m):
                if _maxdist(template, data[j:j + m]) <= r:
                    B += 1.0
                    if j < N - m:
                        if _maxdist(data[i:i + m + 1], data[j:j + m + 1]) <= r:
                            A += 1.0

        if A == 0 or B == 0:
            return float('inf')

        return -np.log(A / B)

    def detrended_fluctuation_analysis(self, data, min_scale=4, max_scale=None):
        """Calculate DFA alpha coefficient"""
        if max_scale is None:
            max_scale = len(data) // 4

        # Integrate the signal
        y = np.cumsum(data - np.mean(data))

        scales = np.logspace(np.log10(min_scale), np.log10(max_scale), 20).astype(int)
        fluctuations = []

        for scale in scales:
            # Divide into windows
            n_windows = len(y) // scale
            if n_windows < 4:
                continue

            F = 0
            for i in range(n_windows):
                start = i * scale
                end = (i + 1) * scale
                segment = y[start:end]

                # Detrend (linear fit)
                t = np.arange(len(segment))
                coeffs = np.polyfit(t, segment, 1)
                trend = np.polyval(coeffs, t)
                detrended = segment - trend

                F += np.sum(detrended**2)

            F = np.sqrt(F / (n_windows * scale))
            fluctuations.append(F)

        if len(fluctuations) < 2:
            return 1.0

        # Fit line in log-log plot
        log_scales = np.log10(scales[:len(fluctuations)])
        log_fluct = np.log10(fluctuations)

        coeffs = np.polyfit(log_scales, log_fluct, 1)
        return coeffs[0]  # Alpha coefficient

# ======================= DEEP LEARNING MODELS =======================

class DeepBioSignalClassifier:
    """Advanced deep learning models for biomedical signal classification"""

    def __init__(self, input_shape, num_classes=2):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.models = {}

    def create_cnn_lstm_model(self, name='cnn_lstm'):
        """CNN-LSTM hybrid model for time series classification"""
        input_layer = Input(shape=self.input_shape)

        # CNN layers for feature extraction
        conv1 = Conv1D(64, kernel_size=3, activation='relu', padding='same')(input_layer)
        conv1 = BatchNormalization()(conv1)
        pool1 = MaxPooling1D(pool_size=2)(conv1)

        conv2 = Conv1D(128, kernel_size=3, activation='relu', padding='same')(pool1)
        conv2 = BatchNormalization()(conv2)
        pool2 = MaxPooling1D(pool_size=2)(conv2)

        conv3 = Conv1D(256, kernel_size=3, activation='relu', padding='same')(pool2)
        conv3 = BatchNormalization()(conv3)
        pool3 = MaxPooling1D(pool_size=2)(pool3)

        # LSTM layers for temporal modeling
        lstm1 = LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)(pool3)
        lstm2 = LSTM(64, dropout=0.3, recurrent_dropout=0.3)(lstm1)

        # Dense layers
        dense1 = Dense(50, activation='relu')(lstm2)
        dropout1 = Dropout(0.5)(dense1)

        # Output layer
        if self.num_classes == 2:
            output = Dense(1, activation='sigmoid')(dropout1)
            loss = 'binary_crossentropy'
        else:
            output = Dense(self.num_classes, activation='softmax')(dropout1)
            loss = 'categorical_crossentropy'

        model = Model(inputs=input_layer, outputs=output)

        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss=loss,
            metrics=['accuracy']
        )

        self.models[name] = model
        return model

    def create_attention_model(self, name='attention'):
        """Attention-based model for biomedical signals"""
        input_layer = Input(shape=self.input_shape)

        # CNN feature extraction
        conv1 = Conv1D(64, 3, activation='relu', padding='same')(input_layer)
        conv1 = BatchNormalization()(conv1)
        conv2 = Conv1D(128, 3, activation='relu', padding='same')(conv1)
        conv2 = BatchNormalization()(conv2)

        # Bidirectional LSTM
        lstm = tf.keras.layers.Bidirectional(LSTM(64, return_sequences=True))(conv2)

        # Attention mechanism
        attention = tf.keras.layers.MultiHeadAttention(num_heads=8, key_dim=64)(lstm, lstm)
        attention = tf.keras.layers.GlobalAveragePooling1D()(attention)

        # Dense layers
        dense1 = Dense(128, activation='relu')(attention)
        dropout1 = Dropout(0.3)(dense1)
        dense2 = Dense(64, activation='relu')(dropout1)
        dropout2 = Dropout(0.3)(dense2)

        # Output
        if self.num_classes == 2:
            output = Dense(1, activation='sigmoid')(dropout2)
            loss = 'binary_crossentropy'
        else:
            output = Dense(self.num_classes, activation='softmax')(dropout2)
            loss = 'categorical_crossentropy'

        model = Model(inputs=input_layer, outputs=output)
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss=loss,
            metrics=['accuracy']
        )

        self.models[name] = model
        return model

    def create_residual_model(self, name='resnet'):
        """ResNet-inspired model for biomedical signals"""
        def residual_block(x, filters, kernel_size=3):
            shortcut = x

            # First conv layer
            x = Conv1D(filters, kernel_size, padding='same')(x)
            x = BatchNormalization()(x)
            x = tf.keras.layers.ReLU()(x)

            # Second conv layer
            x = Conv1D(filters, kernel_size, padding='same')(x)
            x = BatchNormalization()(x)

            # Adjust shortcut if needed
            if shortcut.shape[-1] != filters:
                shortcut = Conv1D(filters, 1, padding='same')(shortcut)
                shortcut = BatchNormalization()(shortcut)

            # Add residual connection
            x = tf.keras.layers.Add()([x, shortcut])
            x = tf.keras.layers.ReLU()(x)

            return x

        input_layer = Input(shape=self.input_shape)

        # Initial conv layer
        x = Conv1D(64, 7, padding='same')(input_layer)
        x = BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = MaxPooling1D(3, strides=2, padding='same')(x)

        # Residual blocks
        x = residual_block(x, 64)
        x = residual_block(x, 64)
        x = MaxPooling1D(2)(x)

        x = residual_block(x, 128)
        x = residual_block(x, 128)
        x = MaxPooling1D(2)(x)

        x = residual_block(x, 256)
        x = residual_block(x, 256)

        # Global average pooling
        x = GlobalAveragePooling1D()(x)

        # Dense layers
        x = Dense(128, activation='relu')(x)
        x = Dropout(0.5)(x)

        # Output
        if self.num_classes == 2:
            output = Dense(1, activation='sigmoid')(x)
            loss = 'binary_crossentropy'
        else:
            output = Dense(self.num_classes, activation='softmax')(x)
            loss = 'categorical_crossentropy'

        model = Model(inputs=input_layer, outputs=output)
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss=loss,
            metrics=['accuracy']
        )

        self.models[name] = model
        return model

# ======================= REAL-TIME PROCESSING =======================

class RealTimeProcessor:
    """Real-time signal processing with streaming capabilities"""

    def __init__(self, buffer_size=1000, sampling_rate=500):
        self.buffer_size = buffer_size
        self.fs = sampling_rate
        self.signal_buffer = np.zeros(buffer_size)
        self.feature_buffer = []
        self.anomaly_threshold = 2.0
        self.processor = AdvancedBioSignalProcessor(sampling_rate)
        self.is_processing = False
        self.data_queue = queue.Queue()

    def add_data_point(self, data_point):
        """Add new data point to circular buffer"""
        self.signal_buffer = np.roll(self.signal_buffer, -1)
        self.signal_buffer[-1] = data_point

        # Add to queue for processing
        self.data_queue.put(data_point)

    def start_real_time_processing(self):
        """Start real-time processing thread"""
        self.is_processing = True
        processing_thread = threading.Thread(target=self._process_stream)
        processing_thread.daemon = True
        processing_thread.start()

    def stop_real_time_processing(self):
        """Stop real-time processing"""
        self.is_processing = False

    def _process_stream(self):
        """Process streaming data"""
        while self.is_processing:
            try:
                # Process every 100 samples (200ms at 500Hz)
                if self.data_queue.qsize() >= 100:
                    # Get latest 100 samples
                    recent_data = []
                    for _ in range(100):
                        if not self.data_queue.empty():
                            recent_data.append(self.data_queue.get())

                    if len(recent_data) == 100:
                        recent_array = np.array(recent_data)

                        # Quick feature extraction
                        features = self._extract_realtime_features(recent_array)
                        self.feature_buffer.append(features)

                        # Keep only last 50 feature vectors
                        if len(self.feature_buffer) > 50:
                            self.feature_buffer.pop(0)

                        # Anomaly detection
                        if len(self.feature_buffer) > 10:
                            self._detect_realtime_anomaly()

            except Exception as e:
                print(f"Real-time processing error: {e}")

    def _extract_realtime_features(self, data):
        """Extract features optimized for real-time processing"""
        features = {}

        # Basic statistical features (fast)
        features['mean'] = np.mean(data)
        features['std'] = np.std(data)
        features['rms'] = np.sqrt(np.mean(data**2))
        features['peak_to_peak'] = np.ptp(data)

        # Spectral features (fast FFT)
        fft_data = np.abs(fft(data))
        features['spectral_energy'] = np.sum(fft_data**2)
        features['spectral_centroid'] = np.sum(np.arange(len(fft_data)) * fft_data) / np.sum(fft_data)

        # Zero crossing rate
        features['zcr'] = len(np.where(np.diff(np.sign(data)))[0]) / len(data)

        return features

    def _detect_realtime_anomaly(self):
        """Real-time anomaly detection"""
        if len(self.feature_buffer) < 10:
            return

        # Convert features to array
        feature_matrix = np.array([[f['mean'], f['std'], f['rms'], f['spectral_energy']]
                                 for f in self.feature_buffer[-10:]])

        # Simple threshold-based anomaly detection
        current_features = feature_matrix[-1]
        mean_features = np.mean(feature_matrix[:-1], axis=0)
        std_features = np.std(feature_matrix[:-1], axis=0)

        # Z-score anomaly detection
        z_scores = np.abs((current_features - mean_features) / (std_features + 1e-6))

        if np.any(z_scores > self.anomaly_threshold):
            self._trigger_anomaly_alert(z_scores)

    def _trigger_anomaly_alert(self, z_scores):
        """Trigger anomaly alert"""
        print(f"ANOMALY DETECTED at {datetime.now()}")
        print(f"Z-scores: {z_scores}")
        # Here you would implement actual alerting (email, SMS, dashboard update, etc.)

# ======================= CLINICAL ANALYSIS SUITE =======================

class ClinicalAnalysisSuite:
    """Advanced clinical analysis with diagnostic capabilities"""

    def __init__(self):
        self.processor = AdvancedBioSignalProcessor()
        self.deep_classifier = None
        self.diagnostic_rules = self._load_diagnostic_rules()
        self.diagnosis_history = []

    def _load_diagnostic_rules(self):
        """Load clinical diagnostic rules"""
        rules = {
            'ecg': {
                'bradycardia': {
                    'heart_rate': lambda hr: hr < 60,
                    'severity': lambda hr: 'severe' if hr < 40 else 'moderate' if hr < 50 else 'mild'
                },
                'tachycardia': {
                    'heart_rate': lambda hr: hr > 100,
                    'severity': lambda hr: 'severe' if

SyntaxError: incomplete input (ipython-input-1-1348264319.py, line 653)

# Task
Complete the project by generating synthetic ECG data, preprocessing it, extracting features, training deep learning models (CNN-LSTM, Attention, ResNet), evaluating the models, implementing anomaly detection, simulating real-time processing, and performing clinical analysis.

## Generate synthetic data

### Subtask:
Create synthetic ECG data with different arrhythmias to simulate a real-world scenario.


**Reasoning**:
Instantiate the `AdvancedBioSignalProcessor` class and generate synthetic ECG data with different arrhythmias as per the instructions.



In [3]:
processor = AdvancedBioSignalProcessor(sampling_rate=500)

ecg_normal, t_normal = processor.generate_synthetic_ecg(duration=60, arrhythmia_type=None)
ecg_af, t_af = processor.generate_synthetic_ecg(duration=60, arrhythmia_type='atrial_fibrillation')
ecg_vt, t_vt = processor.generate_synthetic_ecg(duration=60, arrhythmia_type='ventricular_tachycardia')
ecg_pvc, t_pvc = processor.generate_synthetic_ecg(duration=60, arrhythmia_type='premature_ventricular_contraction')

print("Synthetic ECG data generated for normal, AF, VT, and PVC cases.")

NameError: name 'AdvancedBioSignalProcessor' is not defined

**Reasoning**:
The previous cell failed because the class `AdvancedBioSignalProcessor` was not defined due to a `SyntaxError` in the initial cell. I need to regenerate the full code for the initial cell, fixing the `SyntaxError`, and then proceed with generating the synthetic data.



In [6]:
# Advanced Biomedical Signal Processing Platform
# Complete implementation with deep learning, real-time processing, and clinical analysis

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal, stats
from scipy.fft import fft, fftfreq
import pywt
from PyEMD import EMD, EEMD
import heartpy as hp
import neurokit2 as nk
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv1D, MaxPooling1D, LSTM, GRU, Dense,
                                   Dropout, BatchNormalization, Attention,
                                   Input, Concatenate, GlobalAveragePooling1D)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import streamlit as st
from datetime import datetime, timedelta
import threading
import queue
import warnings
import json
import pickle
from pathlib import Path
warnings.filterwarnings('ignore')

# ======================= ADVANCED SIGNAL PROCESSING CLASS =======================

class AdvancedBioSignalProcessor:
    """Advanced biomedical signal processing with deep learning capabilities"""

    def __init__(self, sampling_rate=500):
        self.fs = sampling_rate
        self.models = {}
        self.scalers = {}
        self.feature_names = []

    def generate_synthetic_ecg(self, duration=60, noise_level=0.05, arrhythmia_type=None):
        """Generate synthetic ECG data with controllable arrhythmias"""
        t = np.linspace(0, duration, int(duration * self.fs))
        ecg = np.zeros_like(t)

        # Base heart rate with variability
        base_hr = 70
        hr_variability = 10 * np.sin(2 * np.pi * 0.1 * t)  # HRV
        instantaneous_hr = base_hr + hr_variability

        # Generate R-R intervals
        rr_intervals = 60 / instantaneous_hr

        # Create ECG morphology
        current_time = 0
        for rr in rr_intervals:
            peak_time = current_time + rr
            if peak_time < duration:
                # P wave
                p_center = peak_time - 0.15
                if p_center > 0:
                    p_wave = 0.1 * np.exp(-((t - p_center) / 0.02)**2)
                    ecg += p_wave

                # QRS complex
                qrs_duration = 0.08
                qrs_kernel = signal.gaussian(int(qrs_duration * self.fs * 2), std=0.01 * self.fs) # Increased kernel size for QRS
                qrs_start = int((peak_time - qrs_duration/2) * self.fs)
                qrs_end = min(qrs_start + len(qrs_kernel), len(ecg))
                if qrs_start >= 0 and qrs_start < len(ecg):
                    ecg[qrs_start:qrs_end] += qrs_kernel[:qrs_end-qrs_start] * 1.2

                # T wave
                t_center = peak_time + 0.3
                if t_center < duration:
                    t_wave = 0.3 * np.exp(-((t - t_center) / 0.1)**2)
                    ecg += t_wave
                current_time = peak_time
            else:
                break # Stop if peak goes beyond duration


        # Add arrhythmias
        if arrhythmia_type == 'atrial_fibrillation':
            # Irregular R-R intervals and absent P waves
            # Simulate R-R interval variability
            base_rr = 60 / base_hr
            rr_af = base_rr + 0.2 * np.random.randn(len(rr_intervals)) # Add random noise to RR
            rr_af = np.maximum(0.4, rr_af) # Ensure positive RR

            current_time_af = 0
            ecg_af = np.zeros_like(t)
            for rr in rr_af:
                peak_time_af = current_time_af + rr
                if peak_time_af < duration:
                    # QRS complex (no clear P wave)
                    qrs_duration = 0.08
                    qrs_kernel = signal.gaussian(int(qrs_duration * self.fs * 2), std=0.01 * self.fs)
                    qrs_start = int((peak_time_af - qrs_duration/2) * self.fs)
                    qrs_end = min(qrs_start + len(qrs_kernel), len(ecg_af))
                    if qrs_start >= 0 and qrs_start < len(ecg_af):
                         ecg_af[qrs_start:qrs_end] += qrs_kernel[:qrs_end-qrs_start] * 1.2
                    # T wave
                    t_center = peak_time_af + 0.3
                    if t_center < duration:
                        t_wave = 0.3 * np.exp(-((t - t_center) / 0.1)**2)
                        ecg_af += t_wave
                    current_time_af = peak_time_af
                else:
                    break
            ecg = ecg_af + 0.1 * np.random.randn(len(t)) # Add fine fibrillatory waves


        elif arrhythmia_type == 'ventricular_tachycardia':
            # Wide QRS complexes at high rate
            vt_frequency = 3.5  # ~210 bpm
            vt_component = 0.8 * np.sin(2 * np.pi * vt_frequency * t)
            # Replace normal rhythm with VT
            ecg = vt_component

        elif arrhythmia_type == 'premature_ventricular_contraction':
            # Add PVCs at random intervals
            pvc_times = np.random.choice(t[int(len(t)*0.1):-int(len(t)*0.1)],
                                       size=int(duration/10), replace=False)
            for pvc_time in pvc_times:
                pvc_idx = int(pvc_time * self.fs)
                if pvc_idx < len(ecg) - int(0.4 * self.fs): # Ensure enough space for PVC
                    # Wide, bizarre QRS
                    pvc_duration = 0.16 # Wider QRS
                    pvc_morphology = signal.gaussian(int(pvc_duration * self.fs * 2), std=0.04 * self.fs) * 2.0 # Increased amplitude and width
                    pvc_start = max(0, pvc_idx - int(pvc_duration/2 * self.fs))
                    pvc_end = min(len(ecg), pvc_start + len(pvc_morphology))
                    ecg[pvc_start:pvc_end] += pvc_morphology[:pvc_end-pvc_start]


        # Add realistic noise
        baseline_wander = 0.1 * np.sin(2 * np.pi * 0.5 * t)
        muscle_noise = noise_level * np.random.randn(len(t))
        powerline_noise = 0.02 * np.sin(2 * np.pi * 50 * t) # Assuming 50 Hz powerline

        ecg = ecg + baseline_wander + muscle_noise + powerline_noise

        return ecg, t

    def advanced_preprocessing(self, signal_data, signal_type='ecg'):
        """Advanced preprocessing pipeline with multiple methods"""

        # Step 1: Baseline correction using empirical mode decomposition
        emd = EEMD()
        imfs = emd.eemd(signal_data) # Let EEMD determine max_imf

        # Remove slow varying components (baseline drift). Usually the last few IMFs + residue
        # A common approach is to sum the first few IMFs
        # Let's assume the first few IMFs contain the signal and high frequency noise
        # We might need to visually inspect IMFs to decide which ones to remove for baseline
        # For simplicity, let's remove the last IMF (residue) and the second to last (often baseline)
        if len(imfs) > 2:
            reconstructed = np.sum(imfs[:-2], axis=0)
        else: # If not enough IMFs, skip EMD baseline correction
            reconstructed = signal_data


        # Step 2: Adaptive filtering
        filtered_signal = reconstructed # Start with EMD result or original if skipped

        if signal_type == 'ecg':
            # ECG-specific filtering
            # Butterworth bandpass filter
            sos = signal.butter(4, [0.5, 40], btype='band', fs=self.fs, output='sos')
            filtered_signal = signal.sosfilt(sos, filtered_signal)

            # Notch filter for powerline interference
            # Check if sampling rate supports 50 or 60 Hz notch
            if self.fs >= 2 * 50:
                b_notch, a_notch = signal.iirnotch(50, 30, fs=self.fs)
                filtered_signal = signal.filtfilt(b_notch, a_notch, filtered_signal)
            elif self.fs >= 2 * 60:
                b_notch, a_notch = signal.iirnotch(60, 30, fs=self.fs)
                filtered_signal = signal.filtfilt(b_notch, a_notch, filtered_signal)


        elif signal_type == 'eeg':
            # EEG-specific filtering
            sos = signal.butter(4, [0.1, 50], btype='band', fs=self.fs, output='sos')
            filtered_signal = signal.sosfilt(sos, filtered_signal)

        # Step 3: Wavelet denoising
        # Choose wavelet and level based on signal characteristics and sampling rate
        # 'db8' level 6 is a common choice but might need tuning
        try:
            coeffs = pywt.wavedec(filtered_signal, 'db8', level=6)

            # Soft thresholding
            # Threshold selection is critical. Using a simple factor of max detail coeff
            sigma = np.median(np.abs(coeffs[-1])) / 0.6745 # Estimate noise std from last detail coeff
            threshold = sigma * np.sqrt(2 * np.log(len(filtered_signal))) # Universal threshold
            # Or a simpler heuristic threshold:
            # threshold = 0.1 * np.max(np.abs(coeffs[-1]))


            coeffs_thresh = [coeffs[0]] + [pywt.threshold(c, threshold, mode='soft') for c in coeffs[1:]] # Don't threshold approximation coeffs
            denoised_signal = pywt.waverec(coeffs_thresh, 'db8')

            # Ensure denoised signal has the same length as original
            if len(denoised_signal) != len(signal_data):
                # This can happen with wavedec/waverec depending on length and level
                # Simple interpolation or truncation might be needed, or adjust level
                # For now, let's just return the filtered signal if lengths don't match
                print(f"Wavelet denoising length mismatch: original={len(signal_data)}, denoised={len(denoised_signal)}. Skipping wavelet step.")
                denoised_signal = filtered_signal

        except Exception as e:
            print(f"Wavelet denoising error: {e}. Skipping wavelet step.")
            denoised_signal = filtered_signal


        return denoised_signal

    def extract_comprehensive_features(self, signal_data, signal_type='ecg'):
        """Extract comprehensive features for machine learning"""
        features = {}

        # Time domain features
        features['mean'] = np.mean(signal_data)
        features['std'] = np.std(signal_data)
        features['var'] = np.var(signal_data)
        features['rms'] = np.sqrt(np.mean(signal_data**2))
        features['skewness'] = stats.skew(signal_data)
        features['kurtosis'] = stats.kurtosis(signal_data)
        features['zero_crossings'] = len(np.where(np.diff(np.sign(signal_data)))[0])

        # Frequency domain features
        # Ensure sufficient data length for Welch
        if len(signal_data) < 1024:
             nperseg = len(signal_data) // 2
        else:
            nperseg = 1024

        if nperseg > 0:
            freqs, psd = signal.welch(signal_data, fs=self.fs, nperseg=nperseg)

            if np.sum(psd) > 1e-12: # Avoid division by zero or very small numbers
                features['spectral_centroid'] = np.sum(freqs * psd) / np.sum(psd)
                features['spectral_bandwidth'] = np.sqrt(np.sum((freqs - features['spectral_centroid'])**2 * psd) / np.sum(psd))

                # Spectral roll-off (85% power)
                cumulative_psd = np.cumsum(psd)
                rolloff_idx = np.where(cumulative_psd >= 0.85 * cumulative_psd[-1])[0]
                if len(rolloff_idx) > 0:
                    features['spectral_rolloff'] = freqs[rolloff_idx[0]]
                else:
                    features['spectral_rolloff'] = freqs[-1] # If 85% not reached, take max freq

            else:
                 features['spectral_centroid'] = 0.0
                 features['spectral_bandwidth'] = 0.0
                 features['spectral_rolloff'] = 0.0


            # Power in different frequency bands
            if signal_type == 'ecg':
                bands = {'low': (0.5, 5), 'mid': (5, 15), 'high': (15, 40)}
            else:  # EEG
                bands = {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 13),
                        'beta': (13, 30), 'gamma': (30, 50)}

            for band_name, (low, high) in bands.items():
                band_mask = (freqs >= low) & (freqs <= high)
                features[f'power_{band_name}'] = np.sum(psd[band_mask])
        else:
            # If not enough data for Welch, set frequency features to 0
            features['spectral_centroid'] = 0.0
            features['spectral_bandwidth'] = 0.0
            features['spectral_rolloff'] = 0.0
            if signal_type == 'ecg':
                bands = {'low': (0.5, 5), 'mid': (5, 15), 'high': (15, 40)}
            else:  # EEG
                bands = {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 13),
                        'beta': (13, 30), 'gamma': (30, 50)}
            for band_name in bands.keys():
                 features[f'power_{band_name}'] = 0.0


        # Wavelet features
        # Ensure sufficient data length for CWT
        if len(signal_data) > 32: # scales range from 1 to 31
            scales = np.arange(1, 32)
            try:
                coefficients, _ = pywt.cwt(signal_data.flatten(), scales, 'morl') # Flatten to ensure 1D
                features['wavelet_energy'] = np.sum(np.abs(coefficients)**2)
                # Avoid log of zero
                features['wavelet_entropy'] = -np.sum((np.abs(coefficients)**2 + 1e-12) * np.log(np.abs(coefficients)**2 + 1e-12))
            except Exception as e:
                print(f"Wavelet feature extraction error: {e}. Setting wavelet features to 0.")
                features['wavelet_energy'] = 0.0
                features['wavelet_entropy'] = 0.0
        else:
            features['wavelet_energy'] = 0.0
            features['wavelet_entropy'] = 0.0


        # Non-linear features
        # Ensure sufficient data length for entropy and DFA
        if len(signal_data) > 100: # Arbitrary minimum length
            std_data = np.std(signal_data)
            r_ap = 0.2 * std_data if std_data > 0 else 0.1 # Avoid r=0
            r_samp = 0.2 * std_data if std_data > 0 else 0.1

            try:
                 features['approximate_entropy'] = self.approximate_entropy(signal_data, m=2, r=r_ap)
            except Exception as e:
                 print(f"Approximate entropy error: {e}. Setting to 0.")
                 features['approximate_entropy'] = 0.0

            try:
                 features['sample_entropy'] = self.sample_entropy(signal_data, m=2, r=r_samp)
            except Exception as e:
                 print(f"Sample entropy error: {e}. Setting to 0.")
                 features['sample_entropy'] = 0.0

            try:
                 features['dfa_alpha'] = self.detrended_fluctuation_analysis(signal_data)
            except Exception as e:
                 print(f"DFA error: {e}. Setting to 1.0 (random walk).")
                 features['dfa_alpha'] = 1.0 # Default for random walk


        else:
            features['approximate_entropy'] = 0.0
            features['sample_entropy'] = 0.0
            features['dfa_alpha'] = 1.0


        if signal_type == 'ecg':
            # ECG-specific features
            ecg_features = self.extract_ecg_features(signal_data)
            features.update(ecg_features)

        # Remove any potential NaN or Inf values
        features = {k: (v if np.isfinite(v) else 0.0) for k, v in features.items()}


        return features

    def extract_ecg_features(self, ecg_signal):
        """Extract ECG-specific clinical features"""
        features = {}

        # R-peak detection using NeuroKit2
        try:
            peaks_nk_info = nk.ecg_peaks(ecg_signal, sampling_rate=self.fs)
            peaks_nk = peaks_nk_info[1]['ECG_R_Peaks']

            if len(peaks_nk) > 2:
                # Heart rate statistics
                # Calculate RR intervals in seconds first, then convert to ms
                rr_intervals_sec = np.diff(peaks_nk) / self.fs
                rr_intervals_ms = rr_intervals_sec * 1000 # ms

                features['mean_hr'] = 60 / np.mean(rr_intervals_sec) if np.mean(rr_intervals_sec) > 0 else 0.0
                features['std_hr'] = np.std(60 / rr_intervals_sec) if np.std(60 / rr_intervals_sec) > 0 else 0.0

                # HRV time domain
                features['mean_rr'] = np.mean(rr_intervals_ms)
                features['std_rr'] = np.std(rr_intervals_ms)
                features['rmssd'] = np.sqrt(np.mean(np.diff(rr_intervals_ms)**2)) if len(rr_intervals_ms) > 1 else 0.0
                features['pnn50'] = np.sum(np.abs(np.diff(rr_intervals_ms)) > 50) / len(rr_intervals_ms) * 100 if len(rr_intervals_ms) > 1 else 0.0

                # HRV frequency domain
                # Need at least ~256 beats for reliable frequency domain HRV (5 mins)
                # For shorter segments, results might be unreliable.
                # Let's set a minimum number of RR intervals, e.g., 50
                if len(rr_intervals_ms) > 50:
                    # Interpolate RR series to fixed rate (e.g., 4 Hz) for PSD
                    time_rr = np.cumsum(rr_intervals_sec)
                    # Ensure time_rr is strictly increasing
                    time_rr = np.unique(time_rr)
                    # Ensure rr_intervals_ms corresponds to unique times
                    unique_rr_intervals_ms = rr_intervals_ms[np.isin(np.cumsum(rr_intervals_sec), time_rr)]

                    if len(time_rr) > 1:
                        time_interp = np.arange(time_rr[0], time_rr[-1], 1/4.0)  # 4 Hz interpolation
                        if len(time_interp) > 1:
                             rr_interp = np.interp(time_interp, time_rr, unique_rr_intervals_ms)

                             # Power spectral density
                             # Ensure enough data for Welch PSD (nperseg should be smaller than data length)
                             nperseg_hrv = min(256, len(rr_interp) // 2)
                             if nperseg_hrv > 0:
                                freqs_hrv, psd_hrv = signal.welch(rr_interp, fs=4.0, nperseg=nperseg_hrv)

                                # HRV frequency bands (standard ranges)
                                vlf_mask = (freqs_hrv >= 0.0033) & (freqs_hrv < 0.04)
                                lf_mask = (freqs_hrv >= 0.04) & (freqs_hrv < 0.15)
                                hf_mask = (freqs_hrv >= 0.15) & (freqs_hrv < 0.4)

                                vlf_power = np.sum(psd_hrv[vlf_mask])
                                lf_power = np.sum(psd_hrv[lf_mask])
                                hf_power = np.sum(psd_hrv[hf_mask])

                                features['vlf_power'] = vlf_power
                                features['lf_power'] = lf_power
                                features['hf_power'] = hf_power
                                features['lf_hf_ratio'] = lf_power / (hf_power + 1e-9) # Add epsilon to avoid division by zero
                             else:
                                 features['vlf_power'] = 0.0
                                 features['lf_power'] = 0.0
                                 features['hf_power'] = 0.0
                                 features['lf_hf_ratio'] = 0.0
                        else:
                             features['vlf_power'] = 0.0
                             features['lf_power'] = 0.0
                             features['hf_power'] = 0.0
                             features['lf_hf_ratio'] = 0.0
                    else:
                         features['vlf_power'] = 0.0
                         features['lf_power'] = 0.0
                         features['hf_power'] = 0.0
                         features['lf_hf_ratio'] = 0.0

                else: # Not enough RR intervals for reliable frequency domain HRV
                    features['vlf_power'] = 0.0
                    features['lf_power'] = 0.0
                    features['hf_power'] = 0.0
                    features['lf_hf_ratio'] = 0.0


                # Morphological features (example: average morphology)
                # This is a simplified approach. A more robust method would align and average beats.
                # Let's extract features from a few representative beats.
                r_peak_indices = peaks_nk[1:-1] # Exclude first/last incomplete beats
                if len(r_peak_indices) > 0:
                    qrs_widths = []
                    r_amplitudes = []
                    q_amplitudes = []
                    s_amplitudes = []

                    # Use NeuroKit2 for wave delineation for more precise morphology
                    try:
                         delineate_info = nk.ecg_delineate(ecg_signal, rpeaks=peaks_nk, sampling_rate=self.fs)
                         delineate_waves = delineate_info[1]

                         for i in range(len(r_peak_indices)):
                             r_idx = r_peak_indices[i]
                             try:
                                 q_idx = delineate_waves['ECG_Q_Peaks'][i] if i < len(delineate_waves['ECG_Q_Peaks']) and not np.isnan(delineate_waves['ECG_Q_Peaks'][i]) else None
                                 s_idx = delineate_waves['ECG_S_Peaks'][i] if i < len(delineate_waves['ECG_S_Peaks']) and not np.isnan(delineate_waves['ECG_S_Peaks'][i]) else None
                                 qrs_on = delineate_waves['ECG_R_Onsets'][i] if i < len(delineate_waves['ECG_R_Onsets']) and not np.isnan(delineate_waves['ECG_R_Onsets'][i]) else None
                                 qrs_off = delineate_waves['ECG_R_Offsets'][i] if i < len(delineate_waves['ECG_R_Offsets']) and not np.isnan(delineate_waves['ECG_R_Offsets'][i]) else None

                                 if qrs_on is not None and qrs_off is not None:
                                     qrs_widths.append((qrs_off - qrs_on) / self.fs * 1000) # QRS width in ms

                                 r_amplitudes.append(ecg_signal[r_idx])
                                 if q_idx is not None:
                                     q_amplitudes.append(ecg_signal[q_idx])
                                 if s_idx is not None:
                                     s_amplitudes.append(ecg_signal[s_idx])

                             except Exception as beat_e:
                                 # Handle potential errors for individual beats
                                 print(f"Error processing beat {i}: {beat_e}")
                                 continue # Skip this beat

                         features['mean_qrs_width'] = np.mean(qrs_widths) if qrs_widths else 0.0
                         features['mean_r_amplitude'] = np.mean(r_amplitudes) if r_amplitudes else 0.0
                         features['mean_q_amplitude'] = np.mean(q_amplitudes) if q_amplitudes else 0.0
                         features['mean_s_amplitude'] = np.mean(s_amplitudes) if s_amplitudes else 0.0

                    except Exception as delineate_e:
                        print(f"NeuroKit2 delineation error: {delineate_e}. Using fallback for morphology.")
                        # Fallback: Simple peak finding for Q/S relative to R
                        # This is less accurate than delineation
                        for r_idx in r_peak_indices:
                            r_amplitudes.append(ecg_signal[r_idx])

                            # Approximate Q and S points
                            q_search_start = max(0, r_idx - int(0.1 * self.fs)) # Search 100ms before R
                            q_search_end = r_idx
                            if q_search_start < q_search_end:
                                q_amp = np.min(ecg_signal[q_search_start:q_search_end])
                                q_amplitudes.append(q_amp)

                            s_search_start = r_idx
                            s_search_end = min(len(ecg_signal), r_idx + int(0.1 * self.fs)) # Search 100ms after R
                            if s_search_start < s_search_end:
                                s_amp = np.min(ecg_signal[s_search_start:s_search_end])
                                s_amplitudes.append(s_amp)

                        features['mean_qrs_width'] = 0.0 # Cannot reliably get width without delineation
                        features['mean_r_amplitude'] = np.mean(r_amplitudes) if r_amplitudes else 0.0
                        features['mean_q_amplitude'] = np.mean(q_amplitudes) if q_amplitudes else 0.0
                        features['mean_s_amplitude'] = np.mean(s_amplitudes) if s_amplitudes else 0.0

                else: # Not enough R peaks for morphological features
                    features['mean_qrs_width'] = 0.0
                    features['mean_r_amplitude'] = 0.0
                    features['mean_q_amplitude'] = 0.0
                    features['mean_s_amplitude'] = 0.0

        except Exception as e:
            print(f"ECG feature extraction error: {e}")
            # Fill with default values if any error occurs
            for key in ['mean_hr', 'std_hr', 'mean_rr', 'std_rr', 'rmssd', 'pnn50',
                       'vlf_power', 'lf_power', 'hf_power', 'lf_hf_ratio',
                       'mean_qrs_width', 'mean_r_amplitude', 'mean_q_amplitude', 'mean_s_amplitude']:
                features[key] = 0.0

        return features

    def approximate_entropy(self, data, m, r):
        """Calculate approximate entropy"""
        # Ensure data is numpy array
        data = np.asarray(data)
        N = len(data)
        if N < m + 2:
            # print(f"Data length {N} too short for ApEn with m={m}")
            return 0.0 # Or handle error appropriately

        def _phi(m_len):
            # Create templates of length m_len
            x = np.array([data[i : i + m_len] for i in range(N - m_len + 1)])
            # Calculate distance between all pairs of templates
            # Using broadcasting for efficiency
            dist = np.max(np.abs(x[:, None, :] - x[None, :, :]), axis=-1)
            # Count pairs within distance r
            C = np.sum(dist <= r, axis=-1)
            # Calculate phi
            # Add a small epsilon to C/len(x) to avoid log(0)
            phi = np.mean(np.log(C / (N - m_len + 1) + 1e-10))
            return phi

        # Ensure r is positive
        if r <= 0:
             # print(f"Radius r must be positive, got {r}")
             return 0.0

        phi_m = _phi(m)
        phi_m_plus_1 = _phi(m + 1)

        # Avoid log(0) or log(very small number) if C/len is 0
        # The _phi function already handles this with 1e-10 epsilon

        return phi_m - phi_m_plus_1


    def sample_entropy(self, data, m, r):
        """Calculate sample entropy"""
        # Ensure data is numpy array
        data = np.asarray(data)
        N = len(data)
        if N < m + 1:
            # print(f"Data length {N} too short for SampEn with m={m}")
            return float('inf') # Conventionally infinity for very short series

        # Ensure r is positive
        if r <= 0:
            # print(f"Radius r must be positive, got {r}")
            return float('inf')

        # Create templates of length m and m+1
        x_m = np.array([data[i : i + m] for i in range(N - m + 1)])
        x_m_plus_1 = np.array([data[i : i + m + 1] for i in range(N - m)])

        # Count pairs within distance r for length m (excluding self-matches)
        B = 0.0
        # Count pairs within distance r for length m+1
        A = 0.0

        # Using nested loops for clarity, broadcasting is more efficient but complex for conditional exclusion
        for i in range(len(x_m)):
            for j in range(len(x_m)):
                if i != j: # Exclude self-matches for B count
                    if np.max(np.abs(x_m[i] - x_m[j])) <= r:
                        B += 1.0

        for i in range(len(x_m_plus_1)):
             for j in range(len(x_m_plus_1)):
                 if i != j: # Exclude self-matches for A count (though definition varies)
                     if np.max(np.abs(x_m_plus_1[i] - x_m_plus_1[j])) <= r:
                         A += 1.0

        # Sample entropy is -log(A/B)
        # Avoid log(0) and division by zero
        if B == 0:
            return float('inf') # Undefined or infinity
        if A == 0:
            return float('inf') # Undefined or infinity

        return -np.log(A / B)


    def detrended_fluctuation_analysis(self, data, min_scale=4, max_scale=None):
        """Calculate DFA alpha coefficient"""
        # Ensure data is numpy array
        data = np.asarray(data)
        N = len(data)

        if N < 100: # Minimum length recommendation varies, e.g., 100 points
             # print(f"Data length {N} too short for DFA")
             return 1.0 # Default to random walk

        if max_scale is None:
            max_scale = N // 4 # Max scale usually N/4 or N/10

        if max_scale < min_scale:
             # print(f"Max scale ({max_scale}) is less than min scale ({min_scale})")
             return 1.0

        # Integrate the signal
        y = np.cumsum(data - np.mean(data))

        # Define scales (window sizes)
        # Use log-spaced scales for better fitting
        scales = np.unique(np.floor(np.logspace(np.log10(min_scale), np.log10(max_scale), 20)).astype(int))
        scales = scales[scales >= min_scale] # Ensure scales are at least min_scale
        scales = scales[scales <= N // 2] # Ensure scales are at most N/2 for reliable windowing

        if len(scales) < 2:
             # print(f"Not enough valid scales ({len(scales)}) for DFA fit")
             return 1.0 # Cannot perform linear fit

        fluctuations = []

        for scale in scales:
            # Divide into windows of size scale
            n_windows = N // scale
            if n_windows == 0:
                continue

            F_scale = 0
            for i in range(n_windows):
                start = i * scale
                end = start + scale
                segment = y[start:end]

                # Detrend using linear fit
                t = np.arange(scale)
                # Avoid issues with segments of length 1
                if scale > 1:
                     coeffs = np.polyfit(t, segment, 1)
                     trend = np.polyval(coeffs, t)
                else:
                     trend = segment[0] # For scale=1, trend is just the value

                detrended = segment - trend

                F_scale += np.sum(detrended**2)

            # Calculate the fluctuation F(n) for the current scale n
            F = np.sqrt(F_scale / (n_windows * scale))
            fluctuations.append(F)

        # Fit line in log-log plot (log10 of scales vs log10 of fluctuations)
        log_scales = np.log10(scales[:len(fluctuations)])
        log_fluct = np.log10(fluctuations)

        # Ensure enough data points for linear fit
        if len(log_scales) < 2:
             # print(f"Not enough points ({len(log_scales)}) for DFA linear fit")
             return 1.0

        # Filter out potential infinite or NaN values if any
        valid_indices = np.isfinite(log_scales) & np.isfinite(log_fluct)
        log_scales = log_scales[valid_indices]
        log_fluct = log_fluct[valid_indices]

        if len(log_scales) < 2:
             # print(f"Not enough valid points ({len(log_scales)}) for DFA linear fit after filtering")
             return 1.0

        try:
            coeffs = np.polyfit(log_scales, log_fluct, 1)
            return coeffs[0]  # Alpha coefficient (slope)
        except Exception as e:
            # print(f"DFA polyfit error: {e}")
            return 1.0 # Return default in case of fitting error


# ======================= DEEP LEARNING MODELS =======================

class DeepBioSignalClassifier:
    """Advanced deep learning models for biomedical signal classification"""

    def __init__(self, input_shape, num_classes=2):
        self.input_shape = input_shape
        self.num_classes = num_classes
        self.models = {}

    def create_cnn_lstm_model(self, name='cnn_lstm'):
        """CNN-LSTM hybrid model for time series classification"""
        input_layer = Input(shape=self.input_shape)

        # CNN layers for feature extraction
        conv1 = Conv1D(64, kernel_size=3, activation='relu', padding='same')(input_layer)
        conv1 = BatchNormalization()(conv1)
        pool1 = MaxPooling1D(pool_size=2)(conv1)

        conv2 = Conv1D(128, kernel_size=3, activation='relu', padding='same')(pool1)
        conv2 = BatchNormalization()(conv2)
        pool2 = MaxPooling1D(pool_size=2)(conv2)

        conv3 = Conv1D(256, kernel_size=3, activation='relu', padding='same')(pool2)
        conv3 = BatchNormalization()(conv3)
        pool3 = MaxPooling1D(pool_size=2)(conv3) # Changed from pool3(pool2)

        # LSTM layers for temporal modeling
        # Ensure the input to LSTM is 3D (batch, timesteps, features)
        # GlobalMaxPooling1D or Flatten can be used before LSTM if needed
        # If MaxPooling1D outputs 3D, it's fine
        # Check shape after pooling: (None, seq_length, filters)
        lstm_input = pool3 # Assuming pool3 output is 3D

        lstm1 = LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.3)(lstm_input)
        lstm2 = LSTM(64, dropout=0.3, recurrent_dropout=0.3)(lstm1)

        # Dense layers
        dense1 = Dense(50, activation='relu')(lstm2)
        dropout1 = Dropout(0.5)(dense1)

        # Output layer
        if self.num_classes == 2:
            output = Dense(1, activation='sigmoid')(dropout1)
            loss = 'binary_crossentropy'
        else:
            output = Dense(self.num_classes, activation='softmax')(dropout1)
            loss = 'categorical_crossentropy'

        model = Model(inputs=input_layer, outputs=output)

        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss=loss,
            metrics=['accuracy']
        )

        self.models[name] = model
        return model

    def create_attention_model(self, name='attention'):
        """Attention-based model for biomedical signals"""
        input_layer = Input(shape=self.input_shape)

        # CNN feature extraction
        conv1 = Conv1D(64, 3, activation='relu', padding='same')(input_layer)
        conv1 = BatchNormalization()(conv1)
        conv2 = Conv1D(128, 3, activation='relu', padding='same')(conv1)
        conv2 = BatchNormalization()(conv2)

        # Bidirectional LSTM
        lstm = tf.keras.layers.Bidirectional(LSTM(64, return_sequences=True))(conv2)

        # Attention mechanism
        # MultiHeadAttention needs query, key, value. Here we use lstm output for all.
        # It outputs (batch, timesteps, key_dim*num_heads)
        attention_output = tf.keras.layers.MultiHeadAttention(num_heads=8, key_dim=64)(lstm, lstm)

        # Need to reduce attention_output to (batch, features) before Dense
        # GlobalAveragePooling1D is a good choice after attention for sequence classification
        attention_pooled = tf.keras.layers.GlobalAveragePooling1D()(attention_output)


        # Dense layers
        dense1 = Dense(128, activation='relu')(attention_pooled)
        dropout1 = Dropout(0.3)(dense1)
        dense2 = Dense(64, activation='relu')(dropout1)
        dropout2 = Dropout(0.3)(dense2)

        # Output
        if self.num_classes == 2:
            output = Dense(1, activation='sigmoid')(dropout2)
            loss = 'binary_crossentropy'
        else:
            output = Dense(self.num_classes, activation='softmax')(dropout2)
            loss = 'categorical_crossentropy'

        model = Model(inputs=input_layer, outputs=output)
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss=loss,
            metrics=['accuracy']
        )

        self.models[name] = model
        return model

    def create_residual_model(self, name='resnet'):
        """ResNet-inspired model for biomedical signals"""
        def residual_block(x, filters, kernel_size=3):
            shortcut = x

            # First conv layer
            x = Conv1D(filters, kernel_size, padding='same')(x)
            x = BatchNormalization()(x)
            x = tf.keras.layers.ReLU()(x)

            # Second conv layer
            x = Conv1D(filters, kernel_size, padding='same')(x)
            x = BatchNormalization()(x)

            # Adjust shortcut if needed (e.g., change in number of filters)
            if shortcut.shape[-1] != filters:
                shortcut = Conv1D(filters, 1, padding='same')(shortcut)
                shortcut = BatchNormalization()(shortcut)

            # Add residual connection
            x = tf.keras.layers.Add()([x, shortcut])
            x = tf.keras.layers.ReLU()(x)

            return x

        input_layer = Input(shape=self.input_shape)

        # Initial conv layer
        x = Conv1D(64, 7, padding='same')(input_layer)
        x = BatchNormalization()(x)
        x = tf.keras.layers.ReLU()(x)
        x = MaxPooling1D(3, strides=2, padding='same')(x)

        # Residual blocks
        x = residual_block(x, 64)
        x = residual_block(x, 64)
        x = MaxPooling1D(2)(x)

        x = residual_block(x, 128)
        x = residual_block(x, 128)
        x = MaxPooling1D(2)(x)

        x = residual_block(x, 256)
        x = residual_block(x, 256)

        # Global average pooling
        x = GlobalAveragePooling1D()(x)

        # Dense layers
        x = Dense(128, activation='relu')(x)
        x = Dropout(0.5)(x)

        # Output
        if self.num_classes == 2:
            output = Dense(1, activation='sigmoid')(x)
            loss = 'binary_crossentropy'
        else:
            output = Dense(self.num_classes, activation='softmax')(x)
            loss = 'categorical_crossentropy'

        model = Model(inputs=input_layer, outputs=output)
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss=loss,
            metrics=['accuracy']
        )

        self.models[name] = model
        return model

# ======================= REAL-TIME PROCESSING =======================

class RealTimeProcessor:
    """Real-time signal processing with streaming capabilities"""

    def __init__(self, buffer_size=1000, sampling_rate=500):
        self.buffer_size = buffer_size
        self.fs = sampling_rate
        self.signal_buffer = np.zeros(buffer_size)
        self.feature_buffer = []
        self.anomaly_threshold = 2.0
        self.processor = AdvancedBioSignalProcessor(sampling_rate)
        self.is_processing = False
        self.data_queue = queue.Queue()

    def add_data_point(self, data_point):
        """Add new data point to circular buffer"""
        self.signal_buffer = np.roll(self.signal_buffer, -1)
        self.signal_buffer[-1] = data_point

        # Add to queue for processing
        self.data_queue.put(data_point)

    def start_real_time_processing(self):
        """Start real-time processing thread"""
        self.is_processing = True
        processing_thread = threading.Thread(target=self._process_stream)
        processing_thread.daemon = True
        processing_thread.start()

    def stop_real_time_processing(self):
        """Stop real-time processing"""
        self.is_processing = False

    def _process_stream(self):
        """Process streaming data"""
        # Process in chunks, e.g., process every 1 second (self.fs samples)
        chunk_size = self.fs
        temp_buffer = []

        while self.is_processing:
             try:
                 # Get data from queue
                 while not self.data_queue.empty():
                      temp_buffer.append(self.data_queue.get())

                 # If we have enough data for a chunk
                 if len(temp_buffer) >= chunk_size:
                      process_chunk = np.array(temp_buffer[:chunk_size])
                      temp_buffer = temp_buffer[chunk_size:] # Keep remaining data

                      # Perform real-time feature extraction and anomaly detection on the chunk
                      features = self._extract_realtime_features(process_chunk)
                      self.feature_buffer.append(features)

                      # Keep only last N feature vectors (e.g., corresponding to last 30 seconds)
                      # N = 30 seconds * features_per_second (1 feature vector per chunk)
                      max_features_in_buffer = 30
                      if len(self.feature_buffer) > max_features_in_buffer:
                         self.feature_buffer.pop(0)

                      # Anomaly detection requires a history of features
                      if len(self.feature_buffer) >= 10: # Need at least 10 chunks history
                         self._detect_realtime_anomaly()

                 # Add a small sleep to prevent the loop from consuming 100% CPU if queue is empty
                 # Or use data_queue.get(timeout=...)
                 if self.data_queue.empty():
                      time.sleep(0.01) # Sleep for 10ms if no data


             except Exception as e:
                print(f"Real-time processing error in _process_stream: {e}")
                # Optionally add a small sleep on error to prevent rapid error looping
                time.sleep(0.1)

    def _extract_realtime_features(self, data):
        """Extract features optimized for real-time processing"""
        features = {}

        # Basic statistical features (fast)
        features['mean'] = np.mean(data)
        features['std'] = np.std(data)
        features['rms'] = np.sqrt(np.mean(data**2))
        features['peak_to_peak'] = np.ptp(data)

        # Spectral features (fast FFT)
        # Avoid FFT on very short data
        if len(data) > 1:
             fft_data = np.abs(fft(data))
             # Take only the positive frequencies
             n_fft = len(data)
             freqs = fftfreq(n_fft, 1/self.fs)[:n_fft//2]
             fft_data_pos = fft_data[:n_fft//2]

             if np.sum(fft_data_pos) > 1e-9: # Avoid division by zero
                 features['spectral_energy'] = np.sum(fft_data_pos**2)
                 # Calculate spectral centroid on positive frequencies
                 features['spectral_centroid'] = np.sum(freqs * fft_data_pos) / np.sum(fft_data_pos)
             else:
                 features['spectral_energy'] = 0.0
                 features['spectral_centroid'] = 0.0
        else:
            features['spectral_energy'] = 0.0
            features['spectral_centroid'] = 0.0


        # Zero crossing rate
        # Ensure data has more than one point
        if len(data) > 1:
             features['zcr'] = len(np.where(np.diff(np.sign(data)))[0]) / (len(data) - 1) # ZCR definition can vary
        else:
             features['zcr'] = 0.0

        # Add other quick features if needed, e.g., simple peak count

        # Ensure no NaNs or Infs in features
        features = {k: (v if np.isfinite(v) else 0.0) for k, v in features.items()}

        return features

    def _detect_realtime_anomaly(self):
        """Real-time anomaly detection"""
        # Need enough history to calculate mean and std
        min_history = 10
        if len(self.feature_buffer) < min_history:
            return

        # Convert recent features to array
        # Select features for anomaly detection - choose stable and informative ones
        feature_keys = ['mean', 'std', 'rms', 'spectral_energy'] # Example features
        try:
             feature_matrix = np.array([[f[key] for key in feature_keys]
                                      for f in self.feature_buffer])

             # Use the last feature vector as the current point
             current_features = feature_matrix[-1, :]

             # Use the history (excluding the current point) to calculate mean and std
             history_features = feature_matrix[:-1, :]

             mean_features = np.mean(history_features, axis=0)
             std_features = np.std(history_features, axis=0)

             # Z-score anomaly detection for each feature
             # Add epsilon to std_features to avoid division by zero
             z_scores = np.abs((current_features - mean_features) / (std_features + 1e-6))

             # Check if any feature's z-score exceeds the threshold
             if np.any(z_scores > self.anomaly_threshold):
                 self._trigger_anomaly_alert(z_scores, feature_keys)

        except Exception as e:
            print(f"Real-time anomaly detection error: {e}")


    def _trigger_anomaly_alert(self, z_scores, feature_keys):
        """Trigger anomaly alert"""
        alert_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"ANOMALY DETECTED at {alert_time}")
        print("Feature Z-scores:")
        for i, key in enumerate(feature_keys):
            print(f"  {key}: {z_scores[i]:.2f}")
        print("-" * 20)
        # Here you would implement actual alerting (log, dashboard update, etc.)
        # Consider adding a cooldown period to avoid spamming alerts


# ======================= CLINICAL ANALYSIS SUITE =======================

class ClinicalAnalysisSuite:
    """Advanced clinical analysis with diagnostic capabilities"""

    def __init__(self):
        self.processor = AdvancedBioSignalProcessor()
        self.deep_classifier = None # To be loaded or trained
        self.diagnostic_rules = self._load_diagnostic_rules()
        self.diagnosis_history = []

    def _load_diagnostic_rules(self):
        """Load clinical diagnostic rules"""
        # This is a placeholder. In a real system, rules would be loaded from a file or database.
        # Rules are simple examples based on common ECG findings.
        rules = {
            'ecg': {
                'bradycardia': {
                    'condition': lambda features: features.get('mean_hr', 0) > 0 and features.get('mean_hr', 0) < 60,
                    'severity': lambda features: 'severe' if features.get('mean_hr', 0) < 40 else 'moderate' if features.get('mean_hr', 0) < 50 else 'mild',
                    'description': "Heart rate is below 60 bpm."
                },
                'tachycardia': {
                    'condition': lambda features: features.get('mean_hr', 0) > 100,
                    'severity': lambda features: 'severe' if features.get('mean_hr', 0) > 150 else 'moderate' if features.get('mean_hr', 0) > 120 else 'mild',
                    'description': "Heart rate is above 100 bpm."
                },
                'atrial_fibrillation_suspicion': {
                    'condition': lambda features: features.get('std_rr', 0) > 50 and features.get('pnn50', 0) < 5, # High RR std, low pNN50 (irregular rhythm)
                    'severity': lambda features: 'high' if features.get('std_rr', 0) > 80 else 'moderate',
                    'description': "Suspicion of Atrial Fibrillation based on irregular rhythm."
                },
                 'pvc_suspicion': {
                     'condition': lambda features: features.get('mean_qrs_width', 0) > 120, # Wide QRS (ms)
                     'severity': lambda features: 'high' if features.get('mean_qrs_width', 0) > 160 else 'moderate',
                     'description': "Suspicion of Premature Ventricular Contractions based on wide QRS complexes."
                 }
                # Add more rules for other conditions (e.g., ST elevation, inverted T waves - requires more detailed morphology features)
            },
            'eeg': {
                'seizure_suspicion': {
                    'condition': lambda features: features.get('power_gamma', 0) > features.get('power_alpha', 0) * 2, # Example: high gamma relative to alpha
                    'severity': lambda features: 'high' if features.get('power_gamma', 0) / (features.get('power_alpha', 1e-9)) > 5 else 'moderate',
                    'description': "Suspicion of seizure activity based on EEG spectral power."
                }
                # Add rules for other EEG patterns (e.g., delta/theta activity for slowing, spikes/sharp waves)
            }
        }
        return rules

    def perform_clinical_analysis(self, signal_data, signal_type='ecg'):
        """Perform clinical analysis based on extracted features and rules"""
        if signal_type not in self.diagnostic_rules:
            print(f"No diagnostic rules defined for signal type: {signal_type}")
            return []

        # 1. Preprocess the signal
        try:
            processed_signal = self.processor.advanced_preprocessing(signal_data, signal_type)
        except Exception as e:
            print(f"Preprocessing failed for clinical analysis: {e}")
            return [] # Cannot proceed without preprocessing

        # 2. Extract comprehensive features
        try:
            features = self.processor.extract_comprehensive_features(processed_signal, signal_type)
            # Ensure features dictionary is valid
            if not features:
                 print("Feature extraction failed.")
                 return []
        except Exception as e:
            print(f"Feature extraction failed for clinical analysis: {e}")
            return []


        # 3. Apply diagnostic rules
        potential_diagnoses = []
        rules = self.diagnostic_rules[signal_type]

        for diagnosis_name, rule in rules.items():
            try:
                if rule['condition'](features):
                    severity = rule.get('severity', lambda f: 'unknown')(features)
                    description = rule.get('description', 'No description available.')
                    potential_diagnoses.append({
                        'diagnosis': diagnosis_name,
                        'severity': severity,
                        'description': description,
                        'features': features # Include features for context
                    })
            except Exception as e:
                print(f"Error applying rule '{diagnosis_name}': {e}")
                # Continue with other rules

        # 4. Integrate Deep Learning Classification (if model is available)
        if self.deep_classifier and signal_type == 'ecg': # Assuming classifier is for ECG
            try:
                # Prepare data for classifier (needs padding/reshaping to input_shape)
                # This is a placeholder - actual implementation needs data windowing and matching model input
                model_input_shape = self.deep_classifier.input_shape
                required_length = model_input_shape[0] # Assuming (sequence_length, 1)

                if len(processed_signal) >= required_length:
                    # Take the last required_length samples
                    classifier_input = processed_signal[-required_length:].reshape(1, required_length, 1)

                    # Scale input if a scaler was used during training
                    # This part requires saving/loading the scaler used for model training
                    # Assuming no scaler for simplicity here, but it's crucial in practice
                    # if self.processor.scalers.get('deep_learning_scaler'):
                    #     classifier_input = self.processor.scalers['deep_learning_scaler'].transform(classifier_input.reshape(-1, 1)).reshape(1, required_length, 1)


                    # Predict
                    predictions = self.deep_classifier.models['cnn_lstm'].predict(classifier_input) # Assuming 'cnn_lstm' name
                    # Process predictions based on model output (sigmoid for binary, softmax for multi-class)
                    # This part depends on how the model was trained and what it predicts
                    # Example for binary classification:
                    if self.deep_classifier.num_classes == 2:
                        probability = predictions[0][0]
                        if probability > 0.5: # Example threshold
                             # Add deep learning prediction as a potential diagnosis
                             potential_diagnoses.append({
                                'diagnosis': 'DL_Anomaly_Detected', # Or specific condition predicted by DL
                                'confidence': float(probability),
                                'description': f"Deep Learning model predicted anomaly with probability {probability:.2f}"
                             })
                    # Example for multi-class classification:
                    # else:
                    #     predicted_class_idx = np.argmax(predictions[0])
                    #     confidence = np.max(predictions[0])
                    #     class_labels = ["Normal", "AF", "VT", "PVC"] # Example labels
                    #     predicted_label = class_labels[predicted_class_idx]
                    #     potential_diagnoses.append({
                    #        'diagnosis': f"DL_Prediction: {predicted_label}",
                    #        'confidence': float(confidence),
                    #        'description': f"Deep Learning model predicted '{predicted_label}' with confidence {confidence:.2f}"
                    #     })


            except Exception as e:
                print(f"Deep learning classification failed: {e}")
                # Continue without DL prediction

        # 5. Store diagnosis history
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        self.diagnosis_history.append({
            'timestamp': timestamp,
            'signal_type': signal_type,
            'diagnoses': potential_diagnoses
        })

        return potential_diagnoses

    def get_diagnosis_history(self):
        """Retrieve stored diagnosis history"""
        return self.diagnosis_history

# ======================= STREAMLIT APP (Placeholder) =======================
# This section is commented out as it requires a Streamlit environment to run
# import streamlit as st
# import plotly.graph_objects as go
# from datetime import datetime
# import time

# def run_streamlit_app():
#     st.title("Advanced Biomedical Signal Processing Platform")

#     # Initialize processor and real-time processor
#     if 'processor' not in st.session_state:
#         st.session_state.processor = AdvancedBioSignalProcessor(sampling_rate=500)
#         st.session_state.realtime_processor = RealTimeProcessor(sampling_rate=500)
#         st.session_state.realtime_processor.start_real_time_processing() # Start real-time processing thread
#         st.session_state.diagnosis_suite = ClinicalAnalysisSuite()


#     processor = st.session_state.processor
#     realtime_processor = st.session_state.realtime_processor
#     diagnosis_suite = st.session_state.diagnosis_suite


#     st.sidebar.header("Settings")
#     signal_type = st.sidebar.selectbox("Signal Type", ['ECG', 'EEG'])
#     sampling_rate = st.sidebar.slider("Sampling Rate (Hz)", 100, 1000, 500)
#     processor.fs = sampling_rate # Update processor sampling rate
#     realtime_processor.fs = sampling_rate # Update real-time processor sampling rate


#     st.sidebar.header("Synthetic Data Generation")
#     data_duration = st.sidebar.slider("Duration (seconds)", 10, 300, 60)
#     arrhythmia_type = st.sidebar.selectbox("Arrhythmia Type", [None, 'atrial_fibrillation', 'ventricular_tachycardia', 'premature_ventricular_contraction'])
#     noise_level = st.sidebar.slider("Noise Level", 0.0, 0.5, 0.05)


#     if st.sidebar.button("Generate Synthetic Data"):
#         with st.spinner("Generating data..."):
#             synthetic_signal, synthetic_time = processor.generate_synthetic_ecg(
#                 duration=data_duration,
#                 noise_level=noise_level,
#                 arrhythmia_type=arrhythmia_type
#             )
#             st.session_state.synthetic_signal = synthetic_signal
#             st.session_state.synthetic_time = synthetic_time
#             st.session_state.current_signal_type = 'ecg' # Store signal type
#         st.success("Synthetic data generated!")


#     st.header("Signal Visualization")
#     if 'synthetic_signal' in st.session_state:
#         fig = go.Figure()
#         fig.add_trace(go.Scatter(x=st.session_state.synthetic_time, y=st.session_state.synthetic_signal, mode='lines', name='Raw Signal'))
#         fig.update_layout(title='Synthetic Signal', xaxis_title='Time (s)', yaxis_title='Amplitude')
#         st.plotly_chart(fig, use_container_width=True)


#         st.header("Signal Preprocessing")
#         if st.button("Apply Advanced Preprocessing"):
#             with st.spinner("Preprocessing..."):
#                 processed_signal = processor.advanced_preprocessing(st.session_state.synthetic_signal, st.session_state.current_signal_type)
#                 st.session_state.processed_signal = processed_signal
#             st.success("Preprocessing complete!")

#         if 'processed_signal' in st.session_state:
#             fig_processed = go.Figure()
#             fig_processed.add_trace(go.Scatter(x=st.session_state.synthetic_time, y=st.session_state.processed_signal, mode='lines', name='Processed Signal'))
#             fig_processed.update_layout(title='Processed Signal', xaxis_title='Time (s)', yaxis_title='Amplitude')
#             st.plotly_chart(fig_processed, use_container_width=True)


#         st.header("Feature Extraction")
#         if st.button("Extract Features"):
#             signal_to_process = st.session_state.processed_signal if 'processed_signal' in st.session_state else st.session_state.synthetic_signal
#             with st.spinner("Extracting features..."):
#                  features = processor.extract_comprehensive_features(signal_to_process, st.session_state.current_signal_type)
#                  st.session_state.extracted_features = features
#             st.success("Feature extraction complete!")
#             st.subheader("Extracted Features:")
#             st.json(st.session_state.extracted_features)


#         st.header("Clinical Analysis")
#         if st.button("Run Clinical Analysis"):
#             signal_for_analysis = st.session_state.processed_signal if 'processed_signal' in st.session_state else st.session_state.synthetic_signal
#             with st.spinner("Performing clinical analysis..."):
#                 diagnoses = diagnosis_suite.perform_clinical_analysis(signal_for_analysis, st.session_state.current_signal_type)
#                 st.session_state.current_diagnoses = diagnoses
#             st.success("Clinical analysis complete!")
#             st.subheader("Potential Diagnoses:")
#             if st.session_state.current_diagnoses:
#                 for diag in st.session_state.current_diagnoses:
#                     st.write(f"- **{diag['diagnosis']}**: {diag.get('description', '')} (Severity: {diag.get('severity', 'N/A')})")
#             else:
#                 st.write("No significant findings based on rules.")


#     st.header("Real-Time Simulation (Placeholder)")
#     st.write("This section simulates real-time data streaming and anomaly detection.")
#     # In a real app, you'd feed data points to realtime_processor.add_data_point()
#     # and display realtime_processor.feature_buffer and anomaly alerts


#     st.sidebar.header("Deep Learning Model")
#     model_name = st.sidebar.selectbox("Choose Model", ['CNN-LSTM', 'Attention', 'ResNet'])
#     # Placeholder for model training/loading
#     st.sidebar.write("Model training/loading functionality not implemented in this placeholder.")


#     # Stop real-time processor thread when app closes or reloads
#     # This is a bit tricky in Streamlit. Using atexit or similar might be needed.
#     # For now, it runs as a daemon thread.
#     # import atexit
#     # atexit.register(realtime_processor.stop_real_time_processing)


# # To run the Streamlit app, save this file and run `streamlit run your_script_name.py`
# # if __name__ == "__main__":
# #     run_streamlit_app()

# # Ensure time is imported for RealTimeProcessor
# import time


ModuleNotFoundError: No module named 'PyEMD'

**Reasoning**:
The previous code block failed because the `PyEMD` library is not installed. I need to install the required library using `pip` and then regenerate the full code block that failed previously.



In [8]:
%pip install PyEMD